## Preparing Embedding files for experiments

- Extract embeddings from ChemBERTa-2 MLM model, save them.

In [1]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from tqdm import tqdm

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

import torch
from lazypredict.Supervised import LazyRegressor

# ---------------------------------------------------------------------------
# Paths and hyperparameters (same as lazy_regression_mlm_min.py)
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
N_SPLITS = 5

BASE = "./"
EMBEDDINGS_DIR = os.path.join(BASE, "Embeddings")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
df = pd.read_csv("../Data/train_df_final_min.csv")
df = df[df['pIC50'] >= 5]

# pd Series to Numpy Array
labels = df['pIC50'].to_numpy()
smiles = df['Canonical_Smiles'].to_numpy()

len(smiles), len(labels)

# === Load pretrained model and tokenizer ===
# MLM model
tokenizer = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MLM')
chemberta = AutoModel.from_pretrained('DeepChem/ChemBERTa-77M-MLM').to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MLM and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# === Dataset Class for Feature Extraction ===
class ChemDataset(Dataset):
    def __init__(self, smiles_list, labels):
        self.smiles_list = smiles_list
        self.labels = labels

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        smi = self.smiles_list[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        inputs = tokenizer(smi, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
        inputs = {key: val.squeeze(0) for key, val in inputs.items()}
        return inputs, label

# === Feature extraction function ===
def extract_chemberta_features(model, dataloader, device):
    """
    Extract ChemBERTa features from SMILES strings using mean pooling.
    """
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting features"):
            inputs, batch_labels = batch
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Run tokenized inputs through ChemBERTa.
            outputs = model(**inputs)

            # Use mean pooling instead of CLS token pooling.
            attention_mask = inputs['attention_mask']
            # Apply the attention mask, then perform mean pooling.
            masked_outputs = outputs.last_hidden_state * attention_mask.unsqueeze(-1)
            pooled = masked_outputs.sum(dim=1) / attention_mask.sum(dim=1, keepdim=True)

            features.append(pooled.cpu().numpy())
            labels.extend(batch_labels.numpy())

    return np.vstack(features), np.array(labels)

# === Training data ===
train_df = df.copy()

print(f"Training set size: {len(smiles)}")

batch_size = 128

# === Build dataset and dataloader ===
train_dataset = ChemDataset(smiles, labels)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

# === Extract ChemBERTa features ===
print("Extracting ChemBERTa features from training set...")
X, y = extract_chemberta_features(chemberta, train_loader, device)

Training set size: 2633
Extracting ChemBERTa features from training set...


Extracting features: 100%|██████████| 21/21 [00:01<00:00, 12.50it/s]


In [4]:
mask_9_10 = (train_df["pIC50"] > 9) & (train_df["pIC50"] <= 10)
mask_13 = (train_df["pIC50"] == 13)

if mask_9_10.sum() > 0 and mask_13.sum() > 0:
    # Pair each sample in 9-10 range with each sample in pIC50=13 range.
    embeddings_9_10 = X[mask_9_10]
    pIC50_9_10 = train_df.loc[mask_9_10, "pIC50"].values

    embeddings_13 = X[mask_13]
    pIC50_13 = train_df.loc[mask_13, "pIC50"].values

    # Build interpolated samples from all pairwise combinations.
    interpolated_embeddings = []
    interpolated_pIC50s = []

    for i in range(len(embeddings_9_10)):
        for j in range(len(embeddings_13)):
            # Average each 9-10 sample with each pIC50=13 sample.
            interpolated_embedding = (embeddings_9_10[i] + embeddings_13[j]) / 2
            interpolated_pIC50 = (pIC50_9_10[i] + pIC50_13[j]) / 2

            interpolated_embeddings.append(interpolated_embedding)
            interpolated_pIC50s.append(interpolated_pIC50)

    # Append interpolated data to the training set.
    X_train_aug = np.vstack([X, np.array(interpolated_embeddings)])
    y_train_aug = np.concatenate([y, interpolated_pIC50s])

    print(f"Generated {len(embeddings_9_10)} × {len(embeddings_13)} = {len(interpolated_embeddings)} interpolated samples")
    print(f"Final feature matrix shape: {X_train_aug.shape}")
    print(f"Final training set size: {len(y_train_aug)}")
else:
    print("Not enough data for interpolation")
print(f"X={X.shape}, y={y.shape}, y in [{y.min():.3f}, {y.max():.3f}]")

Generated 15 × 13 = 195 interpolated samples
Final feature matrix shape: (2828, 384)
Final training set size: 2828
X=(2633, 384), y=(2633,), y in [5.000, 13.000]


In [5]:
X.shape, X_train_aug.shape

((2633, 384), (2828, 384))

### Save files as .npy file

In [7]:
np.save("../Data/MLM_train_embeddings.npy", X)
np.save("../Data/MLM_train_labels.npy", y)
np.save("../Data/MLM_train_aug_embeddings.npy", X_train_aug)
np.save("../Data/MLM_train_aug_labels.npy", y_train_aug)